# Train a model for Fink

This notebook is a template: copy it, then change the cells marked **✏️**.

You bring two things:

1. **your preprocessing**, in `preprocessing/preprocessing.py`: it turns one alert into the features of your model (section 1).
2. **your model**, trained below on the output of this preprocessing.

Everything is logged to MLflow for you: parameters, metrics, training data, model and preprocessing. The model is refused if it would not work in production (wrong features, preprocessing too slow...).

## 0. Connect to MLflow ✏️

In [ ]:
import os
from getpass import getpass

os.environ.setdefault("MLFLOW_TRACKING_URI", "https://mlflow.fink-broker.org")
if "MLFLOW_TRACKING_USERNAME" not in os.environ:
    os.environ["MLFLOW_TRACKING_USERNAME"] = input("MLflow username: ")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("MLflow password: ")

# Name of your model: lowercase letters, digits and dashes.
# Every run of this notebook creates a new version of it.
MODEL_NAME = "my-model"

## 1. Your preprocessing ✏️

Your preprocessing is in **`preprocessing/preprocessing.py`**: open and edit it there. It stays in its own file because this file is what runs in production, on every alert of the stream. This notebook always uses its last saved version.

- `FEATURE_NAMES`: the names of your features, in order;
- `pre_processing(alert)`: takes one alert (a `dict`) and returns one float per name.

Below: its code, then its result on one of your alerts.

In [ ]:
import inspect

import fink_model
from load_alerts import load_alerts

ALERTS = "~/fink-client/ftransfer_ztf_2026-09-28_221528"

prep = fink_model.preprocessing()
print("FEATURE_NAMES =", prep.FEATURE_NAMES, "\n")
print(inspect.getsource(prep.pre_processing))

In [ ]:
alert = next(load_alerts(ALERTS))  # one alert, exactly as received in production
print(alert["objectId"], "->", dict(zip(prep.FEATURE_NAMES, prep.pre_processing(alert))))

## 2. Compute the features of your alerts ✏️

`load_features()` reads your alerts (a folder or a file: `.parquet`, `.avro`, `.jsonl`) and runs your `pre_processing()` on each of them, exactly as in production. It stops if the preprocessing fails on an alert or is too slow for the stream.

- `X`: one row of features per alert, in the order of `FEATURE_NAMES`;
- `info`: `objectId`, `candid`, and the `columns` you ask for, to build your labels.

Use `limit=5000` to try quickly on a part of the alerts.

In [ ]:
X, info = fink_model.load_features(ALERTS, columns=["candidate.classtar"], limit=None)
info.head()

## 3. Your labels ✏️

One label per row of `X`. Build them from `info`, for example from a column of your files (`columns=["tnsclass"]`), or from your own list joined on `objectId`:

```python
labels = pd.read_csv("my_labels.csv")            # objectId,label
y = info.merge(labels, on="objectId", how="left")["label"]
```

**Example only**: here the label is "looks like a star" (SExtractor `classtar` > 0.98, about half of the alerts). Replace it with yours.

In [ ]:
y = (info["candidate.classtar"] > 0.98).astype(int)
y.value_counts()

## 4. Your model ✏️

Any model with a `predict()` method: scikit-learn, XGBoost, LightGBM...

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)

## 5. Train and log to MLflow

Inside `start_run()`, the parameters and the training metrics of your model are logged automatically. Log your own test metrics with `mlflow.log_metrics()`.

`log_model()` checks the model against your preprocessing, then uploads both to MLflow. It prints the version number of your model.

In [ ]:
import mlflow
from sklearn.metrics import accuracy_score, f1_score

with fink_model.start_run(MODEL_NAME):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mlflow.log_metrics({
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
    })

    version = fink_model.log_model(model)

## 6. Deploy

Compare your runs in the MLflow interface (experiment `MODEL_NAME`). Nothing is deployed yet: to build the two containers (preprocessing and model) and run your model on Fink alerts, follow the [Fink AI documentation](https://doc.ztf.fink-broker.org/services/fink_ai/).